In [ ]:
import pickle
from tqdm import tqdm
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
from transformers import AutoFeatureExtractor, ResNetModel, AutoTokenizer, AutoModel
from transformers import ViTFeatureExtractor, ViTModel
from typing import List, Dict
import numpy as np
import os
import pandas as pd
from transformers import pipeline
from PIL import Image
import itertools
import gc

In [ ]:

def chunkify_primary(embs,nft2txs, by):
    """Input:
            dizionario contenente per ogni nft la lista delle sue transazioni 
            metodo che segmenta le transazioni per unità temporali (mensile, trimestrale...)
        
        Output:
            dizionario che contiene, per ogni nft la lista DEI SOLI nft comparsi in quello snapshot
        """
    chunks = {}

    for nft in tqdm(embs, "Generazione chunks"):
        nft_data = nft2txs[nft]
        for tx in nft_data["txs"]:
            segmented_time = by(tx["timestamp"])
            if not segmented_time in chunks:
                chunks[segmented_time] = {}
            if nft not in chunks[segmented_time]:
                chunks[segmented_time][nft] = [tx]
            break
    return {k: v for k, v in sorted(chunks.items(), key=lambda x: x[0])}

import math
def chunkify_all(embs,nft2txs, by):
    """Input:
            dizionario contenente per ogni nft la lista delle sue transazioni 
            metodo che segmenta le transazioni per unità temporali (mensile, trimestrale...)
        
        Output:
            dizionario che contiene, per ogni nft la lista DI TUTTI gli nft comparsi fino a quello snapshot
        """
    chunks = {}

    for nft in tqdm(embs, "Generazione chunks"):
        nft_data = nft2txs[nft]
        for tx in nft_data["txs"]:
            if not math.isnan(float(tx["price"])):
                segmented_time = by(tx["timestamp"])
                if not segmented_time in chunks:
                    chunks[segmented_time] = {}
                if nft not in chunks[segmented_time]:
                    chunks[segmented_time][nft] = [tx]
                else:
                    chunks[segmented_time][nft].append(tx)

    return {k: v for k, v in sorted(chunks.items(), key=lambda x: x[0])}


def loadEmbs(path):
    """"Carica gli embedding nft-aware da un path"""
    EMBS = {
        "train": torch.load(f'{path}/nft2emb_train.pt', map_location="cpu"),
        "val": torch.load(f'{path}/nft2emb_val.pt', map_location="cpu"),
    }
    return EMBS


In [ ]:
path = "/home/user/Progetti/NFT/BASELINE_FOR_ABLATION"
path_images = "../input/bigImageDataset/imgs"

id2img = pickle.load(open("../embeddings/unique2path.pkl", "rb"))


def openImage(path):
    return Image.open(path).convert('RGBA').convert('RGB')


split = "train"
device = "cuda"


# feature_extractor = AutoFeatureExtractor.from_pretrained(
#     "google/vit-base-patch16-224")
# model = AutoModel.from_pretrained("google/vit-base-patch16-224").to(device)
# model.eval()


@torch.no_grad()
def extract_feature(image, pool_index=0):
    """Genera le feature facendo CLS-pooling"""
    inputs = feature_extractor(image, return_tensors="pt")
    for k in inputs:
        inputs[k] = inputs[k].to(device)
    outputs = model(**inputs)
    pooled_output = outputs.last_hidden_state[:, pool_index].reshape(-1)
    model.zero_grad()
    del outputs
    del inputs
    return pooled_output


nft2txs = pickle.load(open(f"../input/nft2txs.pkl", "rb"))


nft2emb = loadEmbs(f"{path}/vit-base-patch16-224")[split]

nft2category = pickle.load(open(f"../input/nft2category.pkl", "rb"))
coll2cat = {}
for nft in nft2emb:
    if eval(nft)[0] not in coll2cat:
        coll2cat[eval(nft)[0]] = nft2category[nft]

nft2emb_aware = {k: torch.from_numpy(v).cpu() for k, v in nft2emb.items()}

if os.path.exists(f"../embeddings/nft2emb_dummyViT-{split}.pkl"):
    nft2emb_unaware = pickle.load(
        open(f"../embeddings/nft2emb_dummyViT-{split}.pkl", 'rb'))
else:
    nft2emb_unaware = {k: extract_feature(openImage(f'{path_images}/{id2img[k]}')).cpu(
    ) for k, v in tqdm(nft2emb.items(), "Running embeddings generation for vit")}
    pickle.dump(nft2emb_unaware, open(
        f"../embeddings/nft2emb_dummyViT-{split}.pkl", 'wb'))

# coll2date=pickle.load(open("../input/coll2created.pkl","rb"))
# coll2date={k:v for k,v in coll2date.items() if v is not None}
# print(len(nft2emb_aware))
# nft2emb_unaware={k:v for k,v in nft2emb_unaware.items() if eval(k)[0] in coll2date}
# nft2emb_aware={k:v for k,v in nft2emb_aware.items() if eval(k)[0]  in coll2date}
# print(len(nft2emb_aware))


In [ ]:
d = nn.CosineSimilarity(dim=1)
import torchmetrics
def getAllDistances(embs):
    DISTS=np.zeros(shape=(len(embs),len(embs)),dtype=np.float16)
    BATCH_SIZE=2048
    PRECISION=4
    embs_vector = torch.cat([embs[nft].unsqueeze(0)
                              for nft in embs]).to(device,dtype=torch.float16)
    embs_key = np.array([nft for nft in embs])       
    
    NFT_LIST=[]
    EMB_LIST=[]   
    N=0
    for i,(n,e) in enumerate(tqdm(embs.items())):
        NFT_LIST.append(n)
        EMB_LIST.append(e)
        if len(NFT_LIST)>=BATCH_SIZE or N+len(NFT_LIST)==len(embs):
            emb=embs_vector[N:N+len(NFT_LIST)]
            dists=torchmetrics.functional.pairwise_cosine_similarity(embs_vector, emb).cpu().numpy().astype(np.float16).T
            #rendo diagonale per risparimare memoria, tanto j,i non serve
            for j in range(len(NFT_LIST)):
                dists[j,len(embs)-(N+j):]=-1
                
            DISTS[N:N+len(NFT_LIST),:]=np.round(dists,PRECISION)
            N+=len(NFT_LIST)
            EMB_LIST=[]
            NFT_LIST=[]

    return DISTS,{nft:i for i,nft in enumerate(embs_key)}
def getPDist(x1,x2):
    # x1 = torch.cat([embs1[nft].unsqueeze(0)
    #                           for nft in embs1]).to(device,dtype=torch.float16)
    # x2 = torch.cat([embs2[nft].unsqueeze(0)
    #                           for nft in embs2]).to(device,dtype=torch.float16)

    DISTS=torch.zeros(shape=(len(x1),len(x2))).to(device)
    BATCH_SIZE=2048
    PRECISION=4

    NFT_LIST=[]
    EMB_LIST=[]   
    N=0
    for i,(e) in enumerate(x1):
        EMB_LIST.append(e)
        if len(EMB_LIST)>=BATCH_SIZE or N+len(EMB_LIST)==len(x1):
            emb=x1[N:N+len(EMB_LIST)]
            dists=torchmetrics.functional.pairwise_cosine_similarity(x2, emb).T
            DISTS[N:N+len(EMB_LIST),:]=torch.round(dists,PRECISION)
            N+=len(EMB_LIST)
            EMB_LIST=[]
    return DISTS

In [ ]:
NFT2SEEN={}
COLL2SEEN={}

for nft in tqdm(nft2emb_unaware):
    txs=nft2txs[nft]
    timestamps=[tx["timestamp"] for tx in txs["txs"]]
    NFT2SEEN[nft]=timestamps[np.argmin(timestamps)]
COLLS=list(set([eval(nft)[0] for nft in NFT2SEEN]))

COLL2NFTS={}
for nft in NFT2SEEN:
    if eval(nft)[0] not in COLL2NFTS:
        COLL2NFTS[eval(nft)[0]]=[]
    COLL2NFTS[eval(nft)[0]].append(nft)

for coll in tqdm(COLL2NFTS):
    timestamps=[NFT2SEEN[nft] for nft in COLL2NFTS[coll]]
    COLL2SEEN[coll]=timestamps[np.argmin(timestamps)]

In [ ]:
# if not os.path.isfile("../input/PAIRWISE_NFT_DISTANCES.csv"):
#     DISTS,embs2key=getAllDistances(nft2emb_unaware)
#     # pd.DataFrame(
    
#     # DISTS,
#     # index=      list(embs2key.keys()),
#     # columns=    list(embs2key.keys())
#     # ).to_csv("../input/PAIRWISE_NFT_DISTANCES.csv",chunksize=1000000)

# else:
#     DISTS=pd.read_csv("../input/PAIRWISE_NFT_DISTANCES.csv",index_col=0)
#     embs2key={e:i for  i,e in enumerate(DISTS.index.values)}
#     DISTS=DISTS.values

# def getDist(DISTS,embs2key,nft1,nft2):
#    id_nft1=embs2key[nft1]
#    id_nft2=embs2key[nft2]
#    d=DISTS[id_nft1][id_nft2]
#    if d==-1:
#       return DISTS[id_nft2][id_nft1]
#    else:
#       return d 


In [ ]:
def segment(timestamp):
    # return str((timestamp.year,(timestamp.week-1)))
    w = timestamp.week-1
    y=timestamp.year
    m=timestamp.month-1

    if w == 52:
        # y += 1
        w = 0
    
    return f"({y}, {w})"
def coll(nft):
    return eval(nft)[0]
NFT2SEEN_SEG={k:segment(v) for k,v in NFT2SEEN.items()}
COLL2SEEN_SEG={k:segment(v) for k,v in COLL2SEEN.items()}

snapshots = chunkify_primary(nft2emb_unaware,nft2txs, by=segment)
snapshots_all = chunkify_all(nft2emb_unaware,nft2txs, by=segment)


seg = "Weekly"

s = list(snapshots.keys())
s = sorted(s, key=lambda x: eval(x)[0]*100+eval(x)[1])
SNAPSHOT2SNAPSHOTIDX={snap:i for i,snap in enumerate(s)}

In [ ]:
def NFT_LevelGraph(snapshot,graph={"E": [], "V": []},old_metadata={}, k=10, min_sim=0):
    #definisco la funzione di similarità utilizzata
    d_fn = nn.CosineSimilarity(dim=1)
    #dizionario conente, per ogni nft la lista delle transazioni ad esso associato fino a questo momento
    new_metadata={}
    #aggiungo al nuovo dizionario i vecchi metadati
    for nft1,tx in old_metadata.items():
        new_metadata[nft1]=old_metadata[nft1]
        
    #aggiungo al nuovo dizionario i nuovi metadati
    for nft1,txs1 in snapshot.items():        #per ogni nft1 in questo snapshot
        if nft1 not in new_metadata:        #aggiungo ai metadati questi nodi
            new_metadata[nft1]=[float(tx["price"]) for tx in  txs1]    
        else:
            new_metadata[nft1].extend([float(tx["price"]) for tx in  txs1])
    #tensore contenente gli embedding degli nft di questo snapshot
    snapshot_embs=snapshot_embs = torch.cat([nft2emb_unaware[nft].unsqueeze(0)for nft in snapshot]).to(device)
    ids_snapshots = np.array(list(snapshot.keys()))

    old_snapshots_embs=None
    ids_old_snapshots=None
    #se nel grafo sono contenuti dei nodi ( non prendo graph["V"] poichè li ci sono i soli nodi che presentano archi)
    if len(new_metadata)>0:
        old_snapshots_embs = torch.cat([nft2emb_unaware[nft].unsqueeze(0)for nft in new_metadata]).to(device)
        ids_old_snapshots = np.array(list(new_metadata.keys()))
    edges_in_graph=set([(src,dest) for src,dest,_ in graph["E"] ])
    for nft1,emb in zip(ids_snapshots,snapshot_embs):        #per ogni nft1 in questo snapshot
        #calcolo le distanze tra gli embedding di nft di questo snapshot e quello di nft1
        #calcolo online
        dists = d_fn(snapshot_embs, emb.unsqueeze(0).to(device))
        #calcolo con distanze precalcolate
        # dists=torch.tensor(DISTS.iloc[[embs2key[k] for k in ids_snapshots]][nft1].values).to(device)
        # dists=[max(DISTS[embs2key[nft1]][k],DISTS[k][nft1]) for k in ids_snapshots]
        # dists=torch.tensor(dists).to(device)
        #mantengo tutti e sole le top k similarità 
        best = (-dists).argsort()[:k]
        #soglio sulla base di min_sim
        if min_sim > 0:
            best = best[dists[best] > min_sim]
        dists=dists.cpu().numpy()
        best=best.cpu().numpy()
        #a questo punto dists contiene la lista delle distanze e best la lista degli indici di  tali similarità 
        #prendo gli identificatori dei best
        selected_nfts=ids_snapshots[best]   
        #popolo sulla base il grafo sulla base di 2 condizioni:
            #   top k (precedentemente calcolati)
            #   sim>min_sim (precedentemente calcolati)
            #   coll(nft1)!=coll(nft2)  le collection devono essrere diverse
            #   time(nft1)>time(nft2)   nft1 deve essere nato dopo nft2 (copiatore >copiato)
        for nft2,d in zip(selected_nfts,dists[best]):
            if eval(nft1)[0]!=eval(nft2)[0] and NFT2SEEN_SEG[nft1]>NFT2SEEN_SEG[nft2] and (nft1,nft2) not in edges_in_graph:       #se la direzione temporale della sorgente-destinazione è quella corretta (copiatore->copiato)
                graph["E"].append({"source":nft1,"target":nft2,"weight":d})
                graph["V"].append(nft1)
                graph["V"].append(nft2)
                edges_in_graph.add((nft1,nft2))
    if ids_old_snapshots is not None:
        for nft1,emb in zip(ids_old_snapshots,old_snapshots_embs):        #per ogni nft1 in questo snapshot
            dists = d_fn(old_snapshots_embs, emb.unsqueeze(0).to(device))
            # dists=[getDist(DISTS,embs2key,nft1,k) for k in ids_old_snapshots]
            # dists=torch.tensor(dists).to(device)
            # dists=torch.tensor(DISTS.iloc[[embs2key[k] for k in ids_old_snapshots]][nft1].values).to(device)
            best = (-dists).argsort()[:k]
            if min_sim > 0:
                best = best[dists[best] > min_sim]
            dists=dists.cpu().numpy()
            best=best.cpu().numpy()   
            selected_nfts=ids_old_snapshots[best]   
            for nft2,d in zip(selected_nfts,dists[best]):
                if eval(nft1)[0]!=eval(nft2)[0] and NFT2SEEN_SEG[nft1]>NFT2SEEN_SEG[nft2] and (nft1,nft2) not in edges_in_graph:       #se la direzione temporale della sorgente-destinazione è quella corretta (copiatore->copiato)
                    graph["E"].append({"source":nft1,"target":nft2,"weight":d})
                    graph["V"].append(nft1)
                    graph["V"].append(nft2)                
                    edges_in_graph.add((nft1,nft2))
    graph["V"]=list(set(graph["V"]))
    COMPLETE_METADATA=[{
            "id":nft,
            "first_seen":NFT2SEEN[nft],
            "first_seen_snap":NFT2SEEN_SEG[nft],
            "#tx":len(new_metadata[nft]),
            "avg_price":np.mean([tx for tx in new_metadata[nft]]),
            "max_price":np.max([tx for tx in new_metadata[nft]]),
            "min_price":np.min([tx for tx in new_metadata[nft]]),
            "std_price":np.std([tx for tx in new_metadata[nft]]),
            "volume":np.sum([tx for tx in new_metadata[nft]]),
            "category":nft2category[nft],
            "collection":eval(nft)[0]
            } 
        for nft in new_metadata]
    return graph,new_metadata,COMPLETE_METADATA


In [ ]:
# resume_from="(2021, 15)"
# E=[dict(e) for i,e in pd.read_csv("../graphsV2/NFTLevel-Pruning_V2-Weekly/(2021, 15)/edge_list.csv").iterrows()]

# V=list(set([e["source"] for e in E]+[e["target"] for e in E]))

# graph={"V":V,"E":E}

# meta__={}
# for id in s[:s.index(resume_from)]:        
#     snapshot=snapshots_all[id]
#     for nft1,txs1 in snapshot.items():
#         if nft1 not in meta__:       
#             meta__[nft1]=[float(tx["price"]) for tx in  txs1]    
#         else:
#              meta__[nft1].extend([float(tx["price"]) for tx in  txs1])
# print(f"|V_all|={len(meta__)}, |V_graph|={len(V)} |E|={len(E)}")

In [ ]:
K=100
MIN_SIM=0.5
graph={"V":[],"E":[]}
meta__={}
pbar=tqdm(s)

# pbar=tqdm(s[s.index(resume_from)+1:])

FOLDER="/home/user/Scrivania"
for snapshot_id in pbar:
    os.makedirs(f"{FOLDER}/{snapshot_id}",exist_ok=True)
    snapshot =snapshots_all[snapshot_id]
    graph,meta__,metadata=NFT_LevelGraph(snapshot,graph=graph,old_metadata=meta__, k=K, min_sim=MIN_SIM)
    edges=graph["E"]
    pd.DataFrame().from_records(edges).drop_duplicates(keep='first').to_csv(f"{FOLDER}/{snapshot_id}/edge_list.csv",index=False)
    pd.DataFrame().from_records(metadata).to_csv(f"{FOLDER}/{snapshot_id}/metadata.csv" ,index=False)
    pbar.set_description(f'Snapshot {snapshot_id} |V_all|={len(metadata)} |V_graph|={len(graph["V"])} |E|={len(graph["E"])}')
    